In [1]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os
from copy import deepcopy
from copy import deepcopy

os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5
from copy import deepcopy

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())

# ====================== 导入依赖 ======================
import sys
# 避免反复执行时 Qt 类重复导入导致崩溃：如果已加载，先删除再导入
if 'draw.pyqt_draw.pyqt_main2' in sys.modules:
    del sys.modules['draw.pyqt_draw.pyqt_main2']

from PyQt5 import QtWidgets
import pyqtgraph as pg
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
import draw.read_snap_xml  as read_snap_xml

# 配置 pyqtgraph：开启抗锯齿，关闭 OpenGL（更稳定）
pg.setConfigOptions(antialias=True)
# pg.setConfigOptions(useOpenGL=False)   # 若驱动或 OpenGL 有问题可显式关闭
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

from config import DATA_DIR,INPUT_DIR


backend (before pyplot): QtAgg
backend (after pyplot): qtagg


In [2]:
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer


In [3]:

from pathlib import Path
# 默认用 "topology_{TIME_2_BUILD}"，也允许用环境变量 TOPOLOGY_VERSION 覆盖
VERSION = os.getenv("TOPOLOGY_VERSION", f"baseline")

RAW_DIR    = Path(INPUT_DIR) / VERSION / "raw"
CONFIG_DIR = Path(INPUT_DIR) /VERSION / "baselie"





FIGURE_DIR    = Path(INPUT_DIR) / VERSION / "figure"



# 若不存在则创建（递归创建上级目录；已存在不报错）

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
import genaric2.tegnode as tegnode
import draw.basic_functio.write2xml as write2xml

# 这里，我们读取到nodes 信息,但是我们需要转化为edge信息，注意这里主要就只有包括inter-edge信息


True


In [4]:
file_in = DATA_DIR
# xml_file = r"DATA_DIR\station_visible_satellites_648_1d_real.xml"
xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"

def slice_group_data(raw_group_data, start, end):
    """
    从 raw_group_data 中裁剪时间区间 [basicSa, end)
    """
    return {
        step: raw_group_data[step]
        for step in range(start, end)
        if step in raw_group_data
    }
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 22006
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, RAW_START, RAW_END)


In [5]:
import genaric2.tegnode as tegnode
import draw.basic_functio.write2xml as write2xml
start_ts =0

end_ts =22005

# 21123-22005
# 可以选择，是看处理后的，还是原始的

#处理后的
# file_path = MODIFY_DIR / f"interplane_links_{start_ts}_{end_ts}.xml"
# rawnodes = write2xml.xml_to_nodes2(file_path, tegnode.tegnode_complete)

#，还是原始的
file_path = RAW_DIR / f"interplane_links_{start_ts}_{end_ts}.xml"
rawnodes = write2xml.xml_to_nodes(file_path, tegnode.tegnode)
#



True


In [6]:
start_ts =0
end_ts = 22005

In [7]:
group_data = slice_group_data(raw_group_data, start_ts, end_ts)


In [8]:
rev_group_data,offset = read_snap_xml.modify_group_data(group_data, N=36, groupid=4)



NameError: name 'group_data' is not defined

In [8]:

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始
# 注意 ，下面是直接将nodes转为edge，因为我们的nodes本身已经完成了冲突检测和处理
import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
all_inter_edge = inter_edge2nodes.trans_nodes2edges(rawnodes,P,N)


In [9]:
# 转化为inter-edge信息后，我们可以通过绘图来初步查看

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []


In [10]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("raw behand")
viewer.resize(1200, 700)
viewer.edges_by_step = all_inter_edge

viewer.show()



In [11]:
import basicSa.fileread.readsatellite as readsatellite
import basicSa.simulator.nodemanager as nodemanager

In [12]:
#sat_dir_path = r'C:\usrspace\mywork\generic\data\648qianfan_xml'  # raw string for file path
sat_dir_path = r'C:\usrspace\mywork\data\648qianfan1d_xml'  # raw string for file path

#sat_dir_path = '/home/yfh/Desktop/Data/onehun_ecef'
satangle = 45
track_angle = 89

BaseRAAN_INCREMENT = 18



SatelliteManager = nodemanager.SatelliteManager()
readsatellite.readsatellite(SatelliteManager,sat_dir_path, satangle, track_angle, P, N, BaseRAAN_INCREMENT, t_start=0, t_end=22005)


(648, 648, [])

In [13]:

import draw.step1.function.linkjson as linkjson
time_idx = 12377
out_json = r"C:\usrspace\mywork\generic\data\grid_snapshot_100.json"


In [14]:
active = []

# 遍历 all_inter_edge[0]
for source, targets in all_inter_edge[time_idx].items():
    for target in targets:
        # 为每个 link 创建一个字典并追加到 links 列表
        active.append((source,  target))
as_str=True
for x in range(P):
    for y in range(N):
        curr = x * N + y
        nxt  = x * N + ((y + 1) % N)
        # 防守式判断，确保字典里确有这些编号的卫星
        if curr in SatelliteManager.satellites and nxt in SatelliteManager.satellites:
            s = str(curr) if as_str else curr
            t = str(nxt)  if as_str else nxt
            active.append((  s,   t))
pending = []

In [15]:
# active = [(0,1), (1,2), {"source":2, "target":3}]
# pending = [(3,4), {"source":5, "target":6}]
linkjson.export_satellites_snapshot_to_json(SatelliteManager, time_idx, out_json,
                                   links_active=active, links_pending=pending)


'C:\\usrspace\\mywork\\generic\\data\\grid_snapshot_100.json'

In [16]:
# 选择导出的分组
selected_groups = [0, 4]
out_path = r"C:\usrspace\mywork\generic\data\grid_highlighted.json"

# 每个分组的颜色
color_map = {
    0: "#ff0000",  # 红色
    1: "#00ff00",  # 绿色
    2: "#0000ff",  # 蓝色
    3: "#FFA500",  # 紫色
    4: "#800080",  # 黄色
    5: "#00FFFF",  # 橙色
    6: "#FFFF00"   # 青色
}



# 调用函数并保存文件
exported_path = linkjson.export_highlighted_to_json(group_data[time_idx], selected_groups, color_map, out_path)

In [17]:
#添加同轨链路

def build_intra_edges_copies(start_ts, end_ts, P, N):
    # 预计算每个节点的左右邻居（tuple 轻量不可变，便于快速构造 set）
    base_neighbors = {
        i * N + j: (i * N + ((j + 1) % N), i * N + ((j - 1) % N))
        for i in range(P) for j in range(N)
    }

    all_intra_edge = {}
    for step in range(start_ts, end_ts):
        # 一次性构造（避免 setdefault & 多次 add 的开销）
        adj = {node: set(neis) for node, neis in base_neighbors.items()}
        all_intra_edge[step] = adj
    return all_intra_edge

# 用法
all_intra_edge = build_intra_edges_copies(start_ts, end_ts, P, N)


In [18]:
# 异轨链路双向化


def make_edges_bidirectional(edge_dict):
    """
    edge_dict: {src: Iterable[dst, ...], ...}
    返回新的 dict[int, set[int]]，不会修改入参
    """
    new_edges = {}
    for src, dsts in edge_dict.items():
        for dst in set(dsts):
            if src == dst:   # 可选：去自环
                continue
            new_edges.setdefault(src, set()).add(dst)
            new_edges.setdefault(dst, set()).add(src)
    return new_edges

# ✅ 生成全新的 used_all_inter_edge（不改 all_inter_edge）
used_all_inter_edge = {
    step: make_edges_bidirectional(edges_t)
    for step, edges_t in all_inter_edge.items()
}


# all_inter_edge=[]
# for step in all_inter_edge:
#     all_inter_edge[step] = make_edges_bidirectional(all_inter_edge[step])


In [19]:
all_edges = {}

for step in range(start_ts, end_ts):
    all_edges[step] = {}
    # 先合并intra_edge
    if step in all_intra_edge:
        for src, dsts in all_intra_edge[step].items():
            all_edges[step].setdefault(src, set()).update(dsts)
    # 再合并inter_edge
    if step in used_all_inter_edge:
        for src, dsts in used_all_inter_edge[step].items():
            all_edges[step].setdefault(src, set()).update(dsts)


In [20]:

viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("main the bidrection of the satellite network")
viewer.resize(1200, 700)
viewer.edges_by_step = all_edges

viewer.show()
_viewer_list.append(viewer)

计算最短路径

In [21]:
from collections import deque
from typing import Dict, Set, Iterable, List, Optional, Any

Adj = Dict[int, Set[int]]

def make_undirected(adj: Adj) -> Adj:
    """
    输入：单步拓扑的邻接表 {u: {v,...}, ...}（可能是有向）
    输出：新的无向邻接表（不会修改入参）
    """
    g: Adj = {}
    for u, vs in adj.items():
        if vs is None:
            continue
        g.setdefault(int(u), set())
        for v in vs:
            if v is None:
                continue
            u2, v2 = int(u), int(v)
            if u2 == v2:
                continue  # 去自环（可选）
            g.setdefault(u2, set()).add(v2)
            g.setdefault(v2, set()).add(u2)
    return g

def bfs_shortest_path(adj: Adj, start: Any, end: Any, *, undirected: bool = True) -> Optional[List[int]]:
    """
    在无权图上求最短路（节点/边权都等于1）。
    - adj: 单步拓扑邻接表 {u: {v,...}, ...}
    - start, end: 起点/终点（int 或可转 int 的字符串）
    - undirected: True=按无向图求解；False=按有向边求解
    返回：最短路径的节点列表，如 [s, ..., t]；不可达返回 None
    """
    s = int(start)
    t = int(end)

    # 准备邻接（是否无向）
    G = make_undirected(adj) if undirected else {int(u): set(map(int, vs)) for u, vs in adj.items()}

    if s == t:
        # 起终点相同，视为零长度路径（若希望至少回传 [s]）
        return [s]

    if s not in G and s not in adj:
        return None
    if t not in G and t not in adj:
        return None

    # 确保起点在字典里（即使它当前没有出边）
    G.setdefault(s, set())

    # BFS
    q = deque([s])
    visited = {s}
    parent: Dict[int, int] = {}

    while q:
        u = q.popleft()
        # 如果某些节点不在 G（仅作为被指向的“孤点”），给它空集合
        for v in G.get(u, set()):
            if v in visited:
                continue
            visited.add(v)
            parent[v] = u
            if v == t:
                # 回溯重建路径
                path = [t]
                while path[-1] != s:
                    path.append(parent[path[-1]])
                path.reverse()
                return path
            q.append(v)

    return None  # 不可达

def path_to_edges(path: List[int]) -> List[tuple]:
    """把节点序列转成边序列 [(u0,u1), (u1,u2), ...]"""
    if not path or len(path) < 2:
        return []
    return list(zip(path[:-1], path[1:]))


In [22]:
# 单步拓扑：all_edges[1] 形如 {src: {dst1, dst2, ...}, ...}
adj_step1 = all_edges[time_idx]

# 求最短路（无向）
path = bfs_shortest_path(adj_step1, start=15, end=193, undirected=True)
print("path:", path)                   # e.g. [12, 47, 118, 345]
print("edges:", path_to_edges(path))   # e.g. [(12,47), (47,118), (118,345)]


path: [15, 51, 50, 49, 85, 121, 157, 193]
edges: [(15, 51), (51, 50), (50, 49), (49, 85), (85, 121), (121, 157), (157, 193)]


注意，上述的拓扑，是只有异轨链路的，并且，在邻接表上，也是单向的。因此，我们实际上要做这几件事情

1.实际网络拓扑是还有同轨链路的，所以，我们还是要加上同轨链路信息

2.在邻接表上，我们需要将所有的边都转化为双向的

接下来，我们可以查看在这段时间内，平均最短路径的变化情况


1. 写成一个专门的函数 2. 在jupter里运行的时候，能够非阻塞 3.我能够选择是否保存图片和pdf

下面是计算平均切换条数

下面将

链路切换与平均最短路径画在一起，这样能够看出链路切换对于平均最短路径的影响


这里是用来查看切换的周期的

这个主要是为了计算周期，就是我要看看，这个变化是随着时间如何变化的